# Gold Layer: Investment Scores

Produces one row per listing with three persona investment scores across normalised signals.
Reads from a single multi-city source table — city is derived from `_FILENAME`, never hardcoded.
Scores are normalised **per city** so each city's listings are ranked independently.
Two signals (`norm_location`, `norm_yield`) are stubs pending confirmation of transport and
property price Silver tables.

**Personas:** Yield Maximiser / Occupancy Optimiser / Quality Host  
**Source:** `AIRBNB_INVESTMENT_DB.SILVER.LISTINGS_CLEANED`  
**Output:** `AIRBNB_INVESTMENT_DB.GOLD.INVESTMENT_SCORES`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
DATABASE      = 'AIRBNB_INVESTMENT_DB'
SILVER_SCHEMA = 'SILVER'
GOLD_SCHEMA   = 'GOLD'
OUTPUT_TABLE  = 'INVESTMENT_SCORES'

In [ ]:
def minmax_norm(series, invert=False):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([0.5] * len(series), index=series.index)
    normed = (series - min_val) / (max_val - min_val)
    return 1 - normed if invert else normed


def load_silver(session):
    df = session.sql("""
        SELECT
            LISTING_ID,
            NEIGHBOURHOOD,
            PROPERTY_TYPE,
            ROOM_TYPE,
            BEDROOMS,
            PRICE,
            AVAILABILITY_365,
            ESTIMATED_OCCUPANCY_L365D,
            ESTIMATED_REVENUE_L365D,
            NUMBER_OF_REVIEWS,
            REVIEW_SCORES_RATING,
            REVIEW_SCORES_CLEANLINESS,
            REVIEW_SCORES_LOCATION,
            REVIEW_SCORES_VALUE,
            HOST_IS_SUPERHOST,
            _FILENAME
        FROM AIRBNB_INVESTMENT_DB.SILVER.LISTINGS_CLEANED
    """).to_pandas()

    city_map = {
        'bristol':            'Bristol',
        'london':             'London',
        'greater_manchester': 'Greater Manchester'
    }
    df['city'] = df['_FILENAME'].str.extract(
        r'raw/inside_airbnb/([^/]+)/')[0].str.lower().map(city_map)

    # Drop rows where key scoring columns are null or zero
    before = len(df)
    df = df[df['ESTIMATED_REVENUE_L365D'].notna()]
    df = df[df['ESTIMATED_REVENUE_L365D'] > 0]
    df = df[df['PRICE'].notna()]
    df = df[df['PRICE'] > 0]

    print(f'Loaded {len(df):,} listings '
          f'(dropped {before - len(df):,} with null/zero revenue)')
    print(df.groupby('city').size())
    return df


def load_geo_data(session):
    df = session.sql("""
        SELECT NEIGHBOURHOOD, AREA_SQKM, _FILENAME
        FROM AIRBNB_INVESTMENT_DB.SILVER.NEIGHBOURHOOD_GEO_CLEANED
        WHERE AREA_SQKM IS NOT NULL AND AREA_SQKM > 0
    """).to_pandas()

    city_map = {
        'bristol':            'Bristol',
        'london':             'London',
        'greater_manchester': 'Greater Manchester'
    }
    df['city'] = df['_FILENAME'].str.extract(
        r'raw/inside_airbnb/([^/]+)/')[0].str.lower().map(city_map)

    print(f'Loaded geo data for {len(df)} neighbourhoods')
    print(df.groupby('city').size())
    return df


def load_poi_data(session):
    df_poi = session.sql("""
        SELECT
            NEIGHBOURHOOD,
            AMENITY_GROUP,
            COUNT(*) AS poi_count
        FROM AIRBNB_INVESTMENT_DB.SILVER.POI_CLEANED
        WHERE AMENITY_GROUP IN (
            'Transport',
            'Attractions & Culture',
            'Dining & Nightlife'
        )
        GROUP BY NEIGHBOURHOOD, AMENITY_GROUP
    """).to_pandas()

    df_pivot = df_poi.pivot_table(
        index='NEIGHBOURHOOD',
        columns='AMENITY_GROUP',
        values='poi_count',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    df_pivot.columns.name = None
    df_pivot = df_pivot.rename(columns={
        'Transport':             'poi_transport',
        'Attractions & Culture': 'poi_attractions',
        'Dining & Nightlife':    'poi_dining',
    })

    for col in ['poi_transport', 'poi_attractions', 'poi_dining']:
        if col not in df_pivot.columns:
            df_pivot[col] = 0

    print(f'Loaded POI data for {len(df_pivot)} neighbourhoods')
    return df_pivot


def load_mart_candidates(session):
    df = session.sql("""
        SELECT
            LISTING_ID,
            ANNUAL_REVENUE,
            AREA_MEDIAN_SALE_PRICE
        FROM AIRBNB_INVESTMENT_DB.GOLD.MART_LISTING_CANDIDATES
        WHERE ANNUAL_REVENUE IS NOT NULL
        AND ANNUAL_REVENUE > 0
        AND AREA_MEDIAN_SALE_PRICE IS NOT NULL
        AND AREA_MEDIAN_SALE_PRICE > 0
    """).to_pandas()

    # Calculate gross yield per listing
    df['gross_yield'] = (
        df['ANNUAL_REVENUE'] / df['AREA_MEDIAN_SALE_PRICE']
    )

    # Cap extreme outliers at 99th percentile
    # to prevent single listings dominating normalisation
    cap = df['gross_yield'].quantile(0.99)
    df['gross_yield'] = df['gross_yield'].clip(upper=cap)

    print(f'Loaded yield data for {len(df):,} listings')
    print(f'Avg gross yield: '
          f'{df["gross_yield"].mean()*100:.2f}%')
    print(f'Max gross yield (capped): '
          f'{df["gross_yield"].max()*100:.2f}%')
    return df[['LISTING_ID', 'gross_yield']]


def compute_investment_scores(df, df_geo, df_poi, df_mart):
    result_dfs = []

    for city in df['city'].unique():
        city_df = df[df['city'] == city].copy()

        # Compute listing density per neighbourhood
        neigh_counts = city_df.groupby('NEIGHBOURHOOD').size().reset_index(
            name='_listing_count')
        city_geo = df_geo[df_geo['city'] == city][['NEIGHBOURHOOD', 'AREA_SQKM']]
        density_df = neigh_counts.merge(city_geo, on='NEIGHBOURHOOD', how='left')
        density_df['listing_density'] = (
            density_df['_listing_count'] / density_df['AREA_SQKM'])

        city_df = city_df.merge(
            density_df[['NEIGHBOURHOOD', 'listing_density']],
            on='NEIGHBOURHOOD',
            how='left'
        )
        city_df['listing_density'] = city_df['listing_density'].fillna(0)

        # Join POI data
        city_poi = df_poi[['NEIGHBOURHOOD', 'poi_transport',
                            'poi_attractions', 'poi_dining'
                           ]].rename(
            columns={'NEIGHBOURHOOD': 'neighbourhood_cleansed'}
        )
        city_df = city_df.merge(
            city_poi,
            left_on='NEIGHBOURHOOD',
            right_on='neighbourhood_cleansed',
            how='left'
        )
        city_df[['poi_transport', 'poi_attractions',
                  'poi_dining']] = city_df[[
            'poi_transport', 'poi_attractions', 'poi_dining'
        ]].fillna(0)

        # Compute weighted POI score
        city_df['poi_score'] = (
            city_df['poi_transport']   * 0.40 +
            city_df['poi_attractions'] * 0.35 +
            city_df['poi_dining']      * 0.25
        )

        # Combine with listing density — POI 70%, density 30%
        city_df['location_score'] = (
            city_df['poi_score']       * 0.70 +
            city_df['listing_density'] * 0.30
        )

        city_df['norm_location'] = minmax_norm(
            city_df['location_score'].fillna(0)
        )

        # Join gross yield data
        city_df = city_df.merge(
            df_mart,
            left_on='LISTING_ID',
            right_on='LISTING_ID',
            how='left'
        )

        # Replace norm_yield stub with real gross yield
        # Fill nulls with 0 before normalising
        # (listings without sale price get 0 yield score)
        city_df['gross_yield'] = city_df['gross_yield'].fillna(0)
        city_df['norm_yield']  = minmax_norm(city_df['gross_yield'])

        # Normalise within city
        city_df['norm_price']      = minmax_norm(
            city_df['PRICE'], invert=True)
        city_df['norm_occupancy']  = minmax_norm(
            city_df['ESTIMATED_OCCUPANCY_L365D'])
        city_df['norm_revenue']    = minmax_norm(
            city_df['ESTIMATED_REVENUE_L365D'])
        city_df['norm_rating']     = minmax_norm(
            city_df['REVIEW_SCORES_RATING'])
        city_df['norm_review_vol'] = minmax_norm(
            city_df['NUMBER_OF_REVIEWS'])

        # Yield Maximiser — revenue and occupancy focused
        city_df['score_yield_maximiser'] = (
            city_df['norm_revenue']    * 0.35 +
            city_df['norm_occupancy']  * 0.25 +
            city_df['norm_price']      * 0.15 +
            city_df['norm_rating']     * 0.15 +
            city_df['norm_review_vol'] * 0.10
        ).mul(100).round(2)

        # Occupancy Optimiser — occupancy and stability focused
        city_df['score_occupancy_optimiser'] = (
            city_df['norm_occupancy']  * 0.40 +
            city_df['norm_revenue']    * 0.20 +
            city_df['norm_rating']     * 0.20 +
            city_df['norm_review_vol'] * 0.10 +
            city_df['norm_price']      * 0.10
        ).mul(100).round(2)

        # Quality Host — rating and guest experience focused
        city_df['score_quality_host'] = (
            city_df['norm_rating']     * 0.45 +
            city_df['norm_review_vol'] * 0.20 +
            city_df['norm_occupancy']  * 0.20 +
            city_df['norm_price']      * 0.15
        ).mul(100).round(2)

        # Both norm_location (POI-enhanced) and norm_yield
        # (gross yield from Land Registry) are now real data
        city_df['score_confidence'] = 'high'
        city_df['computed_at']      = pd.Timestamp.now()
        result_dfs.append(city_df)

    df_out = pd.concat(result_dfs, ignore_index=True)
    print(f'Scores computed for {len(df_out):,} listings '
          f'across {df_out["city"].nunique()} cities')
    return df_out


def write_gold(session, df):
    output_cols = {
        'LISTING_ID':                'listing_id',
        'city':                      'city',
        'NEIGHBOURHOOD':             'neighbourhood_cleansed',
        'PROPERTY_TYPE':             'property_type',
        'ROOM_TYPE':                 'room_type',
        'BEDROOMS':                  'bedrooms',
        'PRICE':                     'price',
        'ESTIMATED_OCCUPANCY_L365D': 'estimated_occupancy',
        'ESTIMATED_REVENUE_L365D':   'estimated_revenue',
        'NUMBER_OF_REVIEWS':         'number_of_reviews',
        'REVIEW_SCORES_RATING':      'review_scores_rating',
        'listing_density':           'listing_density',
        'poi_transport':             'poi_transport',
        'poi_attractions':           'poi_attractions',
        'poi_dining':                'poi_dining',
        'location_score':            'location_score',
        'gross_yield':               'gross_yield',
        'norm_price':                'norm_price',
        'norm_occupancy':            'norm_occupancy',
        'norm_revenue':              'norm_revenue',
        'norm_rating':               'norm_rating',
        'norm_review_vol':           'norm_review_vol',
        'norm_location':             'norm_location',
        'norm_yield':                'norm_yield',
        'score_yield_maximiser':     'score_yield_maximiser',
        'score_occupancy_optimiser': 'score_occupancy_optimiser',
        'score_quality_host':        'score_quality_host',
        'score_confidence':          'score_confidence',
        'computed_at':               'computed_at',
    }
    df_out = df[list(output_cols.keys())].rename(columns=output_cols)

    session.write_pandas(
        df_out,
        OUTPUT_TABLE,
        database=DATABASE,
        schema=GOLD_SCHEMA,
        overwrite=True,
        auto_create_table=True
    )
    print(f'Written {len(df_out):,} rows to GOLD.{OUTPUT_TABLE}')


def validate(session):
    df_val = session.sql("""
        SELECT
            city,
            COUNT(*) AS listing_count,
            ROUND(MIN(score_yield_maximiser), 2) AS min_ym,
            ROUND(MAX(score_yield_maximiser), 2) AS max_ym,
            ROUND(MIN(score_occupancy_optimiser), 2) AS min_oo,
            ROUND(MAX(score_occupancy_optimiser), 2) AS max_oo,
            ROUND(MIN(score_quality_host), 2) AS min_qh,
            ROUND(MAX(score_quality_host), 2) AS max_qh,
            ROUND(AVG(location_score), 4) AS avg_location_score,
            ROUND(AVG(norm_location), 4) AS avg_norm_location,
            ROUND(AVG(norm_yield), 4) AS avg_norm_yield,
            ROUND(AVG(gross_yield), 4) AS avg_gross_yield,
            COUNT(DISTINCT score_confidence) AS confidence_levels,
            COUNT(DISTINCT neighbourhood_cleansed) AS neighbourhoods
        FROM AIRBNB_INVESTMENT_DB.GOLD.INVESTMENT_SCORES
        GROUP BY city
        ORDER BY city
    """).to_pandas()
    print(df_val)
    # Expect: min ~0, max ~100 for all three scores per city

In [ ]:
# --- Orchestration ---
df_silver = load_silver(session)
df_geo    = load_geo_data(session)
df_poi    = load_poi_data(session)
df_mart   = load_mart_candidates(session)
df_scored = compute_investment_scores(
    df_silver, df_geo, df_poi, df_mart)
write_gold(session, df_scored)
validate(session)

In [ ]:
df_val = session.sql('''
    SELECT
        city,
        COUNT(*)                        AS listing_count,
        ROUND(MIN(investment_score), 2) AS min_score,
        ROUND(MAX(investment_score), 2) AS max_score,
        ROUND(AVG(investment_score), 2) AS avg_score,
        COUNT(DISTINCT neighbourhood_cleansed) AS neighbourhoods
    FROM AIRBNB_INVESTMENT_DB.GOLD.INVESTMENT_SCORES
    GROUP BY city
    ORDER BY city
''').to_pandas()
print(df_val)

## Investment Scores Complete

`GOLD.INVESTMENT_SCORES` is ready for Streamlit consumption.  
Each row represents one listing with a composite investment score (0–100).  
Normalisation is applied across all three cities combined so scores are cross-city comparable.  
`score_confidence` is `medium` until `norm_location` and `norm_yield` are replaced with
real Silver data — see `gold_ml_weights.ipynb` for data-driven weight derivation.